In [ ]:
# Fabric notebook source
# METADATA ********************
# Shared utilities for the OHN medallion pipeline.
# Attach this notebook with %run from every transformation notebook.
# ****************************

from datetime import datetime, timezone
from typing import Iterable, Optional

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession, functions as F, Window
from pyspark.sql.types import StringType, TimestampType

spark = SparkSession.builder.getOrCreate()

HIGH_DATE = "9999-12-31 23:59:59"
UNKNOWN_KEY = -1
NOT_APPLICABLE_KEY = -2
LATE_ARRIVING_KEY = -3

AUDIT_COLS = ["_source_system", "_source_file", "_ingest_ts", "_batch_id", "_row_hash"]


# --------------------------------------------------------------------------
# Batch identity
# --------------------------------------------------------------------------
def new_batch_id(source_system: str, entity: str) -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    return f"{source_system}_{entity}_{stamp}"


def now_ts():
    return F.lit(datetime.now(timezone.utc)).cast(TimestampType())


# --------------------------------------------------------------------------
# Hashing
# --------------------------------------------------------------------------
def row_hash(df: DataFrame, columns: Iterable[str], out_col: str = "_row_hash") -> DataFrame:
    """Deterministic SHA-256 over the given columns.

    Nulls are replaced with a sentinel so that (null, 'a') and ('a', null)
    produce different hashes, and every value is trimmed and upper-cased so
    cosmetic source changes do not trigger spurious SCD2 versions.
    """
    parts = [
        F.coalesce(F.upper(F.trim(F.col(c).cast(StringType()))), F.lit("<NULL>"))
        for c in columns
    ]
    return df.withColumn(out_col, F.sha2(F.concat_ws("||", *parts), 256))


def tokenize(col_name: str, salt: str):
    """One-way token for a direct identifier. Salt comes from Key Vault."""
    return F.sha2(F.concat(F.upper(F.trim(F.col(col_name))), F.lit(salt)), 256)


# --------------------------------------------------------------------------
# Watermarks
# --------------------------------------------------------------------------
def get_watermark(entity: str, default: str = "1900-01-01 00:00:00") -> str:
    df = (
        spark.table("lh_bronze.ctl_watermark")
        .filter(F.col("entity_name") == entity)
        .orderBy(F.col("updated_ts").desc())
        .limit(1)
    )
    row = df.collect()
    return row[0]["watermark_value"] if row else default


def set_watermark(entity: str, value: str, batch_id: str) -> None:
    payload = spark.createDataFrame(
        [(entity, str(value), batch_id, datetime.now(timezone.utc))],
        "entity_name string, watermark_value string, batch_id string, updated_ts timestamp",
    )
    tgt = DeltaTable.forName(spark, "lh_bronze.ctl_watermark")
    (
        tgt.alias("t")
        .merge(payload.alias("s"), "t.entity_name = s.entity_name")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


# --------------------------------------------------------------------------
# Standardization helpers
# --------------------------------------------------------------------------
def std_string(col):
    """Trim, collapse internal whitespace, upper-case, empty string to null."""
    cleaned = F.upper(F.trim(F.regexp_replace(col, r"\s+", " ")))
    return F.when(cleaned == "", None).otherwise(cleaned)


def std_postal_code(col):
    """Canadian postal code to A1A1A1, null when the format is invalid."""
    compact = F.upper(F.regexp_replace(col, r"[^A-Za-z0-9]", ""))
    return F.when(compact.rlike(r"^[A-Z]\d[A-Z]\d[A-Z]\d$"), compact).otherwise(None)


def std_phone(col, country_code: str = "+1"):
    digits = F.regexp_replace(col, r"\D", "")
    ten = F.when(F.length(digits) == 11, F.expr("substring(_d, 2, 10)"))
    return (
        F.when(F.length(digits) == 10, F.concat(F.lit(country_code), digits))
        .when(F.length(digits) == 11, F.concat(F.lit("+"), digits))
        .otherwise(None)
    )


def age_band(birth_date_col, as_of_col=None):
    as_of = as_of_col if as_of_col is not None else F.current_date()
    age = F.floor(F.months_between(as_of, birth_date_col) / 12)
    return (
        F.when(age.isNull(), F.lit("Unknown"))
        .when(age < 18, F.lit("0-17"))
        .when(age < 35, F.lit("18-34"))
        .when(age < 50, F.lit("35-49"))
        .when(age < 65, F.lit("50-64"))
        .when(age < 75, F.lit("65-74"))
        .when(age < 85, F.lit("75-84"))
        .otherwise(F.lit("85+"))
    )


def validate_ontario_hcn(col):
    """Ontario health card: 10 digits with a mod-10 (Luhn) check digit.

    Returns a boolean column. Synthetic data must still satisfy the format so
    that validation logic is exercised end to end.
    """
    digits = F.regexp_replace(col, r"\D", "")
    valid_len = F.length(digits) == 10
    luhn = F.expr(
        """
        aggregate(
            transform(
                sequence(0, 9),
                i -> CASE WHEN i % 2 = 0
                          THEN CASE WHEN cast(substring(_hcn_digits, i + 1, 1) as int) * 2 > 9
                                    THEN cast(substring(_hcn_digits, i + 1, 1) as int) * 2 - 9
                                    ELSE cast(substring(_hcn_digits, i + 1, 1) as int) * 2 END
                          ELSE cast(substring(_hcn_digits, i + 1, 1) as int) END
            ),
            0, (acc, x) -> acc + x
        ) % 10 = 0
        """
    )
    return valid_len & luhn


def map_code(df: DataFrame, domain: str, source_system: str,
             source_col: str, target_col: str,
             mapping_table: str = "lh_silver.ref_code_mapping") -> DataFrame:
    """Left-join a coded column to the standard code, defaulting to UNKNOWN.

    Unmapped codes are not silently dropped; the caller writes them to
    governance.unmapped_codes so a steward can extend the mapping.
    """
    m = (
        spark.table(mapping_table)
        .filter((F.col("domain") == domain) & (F.col("source_system") == source_system) & F.col("is_active"))
        .select(F.col("source_code").alias("_src_code"), F.col("standard_code").alias("_std_code"))
    )
    joined = df.join(m, std_string(F.col(source_col)) == F.col("_src_code"), "left")
    return joined.withColumn(target_col, F.coalesce(F.col("_std_code"), F.lit("UNKNOWN"))).drop("_src_code", "_std_code")


# --------------------------------------------------------------------------
# Dimension key lookup
# --------------------------------------------------------------------------
def lookup_dim_key(fact: DataFrame, dim_table: str, business_key_col: str,
                   dim_business_key: str, dim_key_col: str,
                   event_ts_col: Optional[str] = None,
                   out_col: Optional[str] = None) -> DataFrame:
    """Resolve a surrogate key.

    When event_ts_col is supplied the lookup is temporally correct against an
    SCD2 dimension: the version of the member that was in effect at the moment
    the event happened is used, not today's version. Unresolved keys fall back
    to the unknown member rather than dropping the fact row.
    """
    out_col = out_col or dim_key_col
    dim = spark.table(dim_table)

    if event_ts_col:
        d = dim.select(
            F.col(dim_business_key).alias("_bk"),
            F.col(dim_key_col).alias("_sk"),
            F.col("effective_from_ts").alias("_from"),
            F.col("effective_to_ts").alias("_to"),
        )
        cond = (
            (std_string(fact[business_key_col]) == F.col("_bk"))
            & (fact[event_ts_col] >= F.col("_from"))
            & (fact[event_ts_col] < F.col("_to"))
        )
    else:
        d = dim.filter(F.col("is_current")).select(
            F.col(dim_business_key).alias("_bk"), F.col(dim_key_col).alias("_sk")
        )
        cond = std_string(fact[business_key_col]) == F.col("_bk")

    return (
        fact.join(d, cond, "left")
        .withColumn(out_col, F.coalesce(F.col("_sk"), F.lit(UNKNOWN_KEY)))
        .drop("_bk", "_sk", "_from", "_to")
    )


# --------------------------------------------------------------------------
# SCD Type 2 merge
# --------------------------------------------------------------------------
def scd2_merge(source: DataFrame, target_table: str, business_key: str,
               tracked_cols: Iterable[str], key_col: str, batch_id: str) -> dict:
    """Two-phase SCD Type 2 upsert on a Delta table.

    Phase 1 expires the current row whose row_hash differs from the incoming
    row. Phase 2 inserts the new version plus any brand-new members. Delta
    MERGE cannot both update an existing row and insert its replacement in one
    pass, which is why this is split.
    """
    src = row_hash(source, tracked_cols).withColumn("_batch_id", F.lit(batch_id))
    tgt = DeltaTable.forName(spark, target_table)
    ts = datetime.now(timezone.utc)

    # Phase 1 -------------------------------------------------------------
    (
        tgt.alias("t")
        .merge(src.alias("s"), f"t.{business_key} = s.{business_key} AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t._row_hash <> s._row_hash",
            set={
                "effective_to_ts": F.lit(ts),
                "is_current": F.lit(False),
                "updated_ts": F.lit(ts),
            },
        )
        .execute()
    )

    # Phase 2 -------------------------------------------------------------
    current = (
        spark.table(target_table)
        .filter(F.col("is_current"))
        .select(F.col(business_key).alias("_t_bk"), F.col("_row_hash").alias("_t_hash"))
    )
    to_insert = (
        src.join(current, src[business_key] == F.col("_t_bk"), "left")
        .filter(F.col("_t_bk").isNull() | (F.col("_t_hash") != F.col("_row_hash")))
        .drop("_t_bk", "_t_hash")
        .withColumn("effective_from_ts", F.lit(ts))
        .withColumn("effective_to_ts", F.lit(HIGH_DATE).cast(TimestampType()))
        .withColumn("is_current", F.lit(True))
        .withColumn("created_ts", F.lit(ts))
        .withColumn("updated_ts", F.lit(ts))
    )

    max_key = spark.table(target_table).agg(F.coalesce(F.max(key_col), F.lit(0)).alias("m")).collect()[0]["m"]
    w = Window.orderBy(F.col(business_key))
    to_insert = to_insert.withColumn(key_col, F.row_number().over(w) + F.lit(max_key))

    inserted = to_insert.count()
    to_insert.write.format("delta").mode("append").option("mergeSchema", "false").saveAsTable(target_table)

    return {"target": target_table, "versions_inserted": inserted, "batch_id": batch_id}


# --------------------------------------------------------------------------
# Logging
# --------------------------------------------------------------------------
def log_batch(batch_id: str, pipeline: str, source_system: str, target_table: str,
              status: str, rows_read: int = 0, rows_written: int = 0,
              error_message: Optional[str] = None) -> None:
    ts = datetime.now(timezone.utc)
    row = spark.createDataFrame(
        [(batch_id, pipeline, source_system, target_table, status,
          int(rows_read), int(rows_written), error_message, ts)],
        """batch_id string, pipeline_name string, source_system string, target_table string,
           status string, rows_read long, rows_written long, error_message string, log_ts timestamp""",
    )
    row.write.format("delta").mode("append").saveAsTable("lh_bronze.ctl_batch_log")